## Machine Learning Notebook Week-3

### Table of Contents

##### 1. Ensemble Learning 
##### 2. Types of Ensemble Learning 
##### 3. Bagged Models - Random Forest Model
##### 4. Stacked Models 
##### 5. Boosting Models - AdaBoost, Gradient Boost, XGBoost
##### 6. Voting Classifier

##### Ensemble Learning
It is a technique in machine learning where multiple models (learners) are combined to solve a problem and improve overall performance.
The main idea is that a group of weak or average models can perform better together than any single model alone.

In simple words:
Ensemble Learning means combining the predictions of many models to get a more accurate and reliable final result.

##### Types of Ensemble Learning 

###### 1. Bagging : 
Bagging is an ensemble technique that builds multiple models on different random subsets of the training data (created through bootstrapping) and then averages their predictions (for regression) or takes a majority vote (for classification).
It helps reduce variance and overfitting

###### 2. Boosting : 
Boosting is an ensemble method that builds models sequentially, where each new model focuses on correcting the errors made by the previous ones.
It gives more weight to misclassified observations so that the next model learns them better, improving overall accuracy.

###### 3. Stacking :
Stacking is an ensemble technique where multiple different models (base learners) are trained on same data, and their outputs are combined by a meta-model (also called a blender) that learns the best way to combine them.

##### Bagging Models

##### Random Forest Model

Random Forest is an ensemble learning algorithm that combines multiple decision trees to make more accurate and stable predictions.
It builds several trees during training and takes the majority vote (for classification) or average prediction (for regression) from all the trees to produce the final output.

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt

In [24]:
df = pd.read_csv('loan_default.csv')
cat_data = df[['Gender', 'Occupation','Dependent','Income']]
num_data = df[['Cbill_Score','Loan_Amount_Lakhs','Age']]
target = df[['Loan_Default']]
encoded_cat_data = pd.get_dummies(cat_data,drop_first=True,dtype='int')
final_df = pd.concat([num_data,encoded_cat_data,target],axis=1)
from sklearn.model_selection import train_test_split
X = final_df.drop(columns='Loan_Default')
y = final_df['Loan_Default']
xtrain,xtest,ytrain,ytest = train_test_split(X,y,train_size=0.7,test_size=0.3,random_state=133)

##### Random Forest Classifier

In [25]:
from sklearn.ensemble import RandomForestClassifier

In [26]:
rf_class = RandomForestClassifier()
rf_class.fit(xtrain,ytrain)
rf_class_pred = rf_class.predict(xtest) 

##### Model Evaluation

In [27]:
from sklearn.metrics import confusion_matrix,classification_report

In [28]:
print(confusion_matrix(ytest,rf_class_pred))
print(classification_report(ytest,rf_class_pred))

[[223  20]
 [ 51   6]]
              precision    recall  f1-score   support

          No       0.81      0.92      0.86       243
         Yes       0.23      0.11      0.14        57

    accuracy                           0.76       300
   macro avg       0.52      0.51      0.50       300
weighted avg       0.70      0.76      0.73       300



##### Hyper parameter Tuning

In [31]:
from sklearn.model_selection import GridSearchCV

In [33]:
rf_class = RandomForestClassifier()
params = {
    'n_estimators': [100, 200, 500],
    'max_depth': [5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'max_features': ['sqrt', 'log2'],
    'criterion': ['gini', 'entropy']}

rf_grid = GridSearchCV(estimator=rf_class,param_grid=params,cv=3,n_jobs=-1,scoring='f1_weighted')
rf_grid.fit(X,y)

GridSearchCV(cv=3, estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [5, 10, 20],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 5],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 500]},
             scoring='f1_weighted')

In [35]:
rf_grid.best_params_

{'criterion': 'entropy',
 'max_depth': 20,
 'max_features': 'log2',
 'min_samples_leaf': 1,
 'min_samples_split': 5,
 'n_estimators': 500}

##### RF Tuned Model

In [38]:
rf_tuned_model = RandomForestClassifier(criterion= 'entropy',max_depth= 20,max_features= 'log2',min_samples_leaf= 1,min_samples_split= 5,n_estimators = 500)
rf_tuned_model.fit(xtrain,ytrain)
rf_tuned_pred = rf_tuned_model.predict(xtest)

##### Model Evaluation 

In [39]:
print(confusion_matrix(ytest,rf_tuned_pred))
print(classification_report(ytest,rf_tuned_pred))

[[230  13]
 [ 51   6]]
              precision    recall  f1-score   support

          No       0.82      0.95      0.88       243
         Yes       0.32      0.11      0.16        57

    accuracy                           0.79       300
   macro avg       0.57      0.53      0.52       300
weighted avg       0.72      0.79      0.74       300



In [17]:
df = pd.read_csv('Salary.csv')
cat_data = df.select_dtypes(exclude = np.number)
num_data = df.select_dtypes(include = np.number)
encoded_cat_data = pd.get_dummies(cat_data,drop_first=True,dtype='int')
final_df = pd.concat([num_data,encoded_cat_data],axis=1)
x = final_df.drop(columns='Salary_Rupees')
y = final_df['Salary_Rupees']
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,train_size=0.7,test_size=0.3)

##### Random Forest Regressor

In [15]:
from sklearn.ensemble import RandomForestRegressor

In [20]:
rf_reg = RandomForestRegressor()
rf_reg.fit(x_train,y_train)
rf_reg_pred = rf_reg.predict(x_test)

##### Model Evaluation

In [22]:
from sklearn.metrics import mean_squared_error

In [23]:
print('RMSE:',np.sqrt(mean_squared_error(y_test,rf_reg_pred)))

RMSE: 11449.272625189002


##### KNN Classifier Bagged Model

In [42]:
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier

In [44]:
knn = KNeighborsClassifier(n_neighbors=5)

bagged_knn = BaggingClassifier(
    estimator=knn,
    n_estimators=50,        # number of KNN models
    max_samples=0.8,        # 80% of data for each bootstrap sample
    max_features=1.0,       # use all features
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

bagged_knn.fit(xtrain, ytrain)


BaggingClassifier(estimator=KNeighborsClassifier(), max_samples=0.8,
                  n_estimators=50, n_jobs=-1, random_state=42)

In [45]:
bagged_knn_pred = bagged_knn.predict(xtest)

##### Model Evaluation

In [46]:
print(confusion_matrix(ytest,bagged_knn_pred))
print(classification_report(ytest,bagged_knn_pred))

[[228  15]
 [ 54   3]]
              precision    recall  f1-score   support

          No       0.81      0.94      0.87       243
         Yes       0.17      0.05      0.08        57

    accuracy                           0.77       300
   macro avg       0.49      0.50      0.47       300
weighted avg       0.69      0.77      0.72       300



##### Boosting: 

##### AdaBoost Model 

In [62]:
from sklearn.ensemble import AdaBoostClassifier

In [63]:
ada_boost = AdaBoostClassifier()
ada_boost.fit(xtrain,ytrain)
ada_pred = ada_boost.predict(xtest)

##### Model Evaluation

In [64]:
print(confusion_matrix(ytest,ada_pred))
print(classification_report(ytest,ada_pred))

[[242   1]
 [ 56   1]]
              precision    recall  f1-score   support

          No       0.81      1.00      0.89       243
         Yes       0.50      0.02      0.03        57

    accuracy                           0.81       300
   macro avg       0.66      0.51      0.46       300
weighted avg       0.75      0.81      0.73       300



##### Gradient Boost Model

In [65]:
from sklearn.ensemble import GradientBoostingClassifier

In [66]:
GB_class = GradientBoostingClassifier()
GB_class.fit(xtrain,ytrain)
GB_pred = GB_class.predict(xtest)

##### Model Evaluation

In [67]:
print(confusion_matrix(ytest,GB_pred))
print(classification_report(ytest,GB_pred))

[[226  17]
 [ 54   3]]
              precision    recall  f1-score   support

          No       0.81      0.93      0.86       243
         Yes       0.15      0.05      0.08        57

    accuracy                           0.76       300
   macro avg       0.48      0.49      0.47       300
weighted avg       0.68      0.76      0.71       300



##### XGBoost Model

In [70]:
#pip install xgboost

In [74]:
from xgboost import XGBClassifier
ytrain_1 = ytrain.replace(to_replace=['No','Yes'],value=[0,1])

C:\Users\cenar\AppData\Local\Temp\ipykernel_27464\2940353916.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ytrain_1 = ytrain.replace(to_replace=['No','Yes'],value=[0,1])


In [75]:
xgb_model = XGBClassifier(n_estimators=100,max_depth=5,learning_rate=0.1,random_state=42)
xgb_model.fit(xtrain, ytrain_1)
xgb_pred = xgb_model.predict(xtest)

##### Model Evaluation

In [77]:
ytest_1 = ytest.replace(to_replace=['No','Yes'],value=[0,1])
print(confusion_matrix(ytest_1,xgb_pred))
print(classification_report(ytest_1,xgb_pred))

[[221  22]
 [ 47  10]]
              precision    recall  f1-score   support

           0       0.82      0.91      0.86       243
           1       0.31      0.18      0.22        57

    accuracy                           0.77       300
   macro avg       0.57      0.54      0.54       300
weighted avg       0.73      0.77      0.74       300



C:\Users\cenar\AppData\Local\Temp\ipykernel_27464\1700549933.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ytest_1 = ytest.replace(to_replace=['No','Yes'],value=[0,1])


##### Stacked Model: 

In [55]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

In [56]:
base_learners = [('knn', KNeighborsClassifier(n_neighbors=5)),
    ('dt', DecisionTreeClassifier(random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42))]

In [57]:
meta_model = LogisticRegression(random_state=42)

In [59]:
stacked_model = StackingClassifier(estimators=base_learners,final_estimator=meta_model,cv=5,n_jobs=-1)
stacked_model.fit(xtrain,ytrain)
stacked_pred = stacked_model.predict(xtest)

##### Model Evaluation

In [61]:
print(confusion_matrix(ytest,stacked_pred))
print(classification_report(ytest,stacked_pred))

[[243   0]
 [ 57   0]]
              precision    recall  f1-score   support

          No       0.81      1.00      0.90       243
         Yes       0.00      0.00      0.00        57

    accuracy                           0.81       300
   macro avg       0.41      0.50      0.45       300
weighted avg       0.66      0.81      0.72       300



C:\Users\cenar\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\cenar\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\cenar\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


##### Voting Classifier

A Voting Classifier is an ensemble learning method that combines predictions from multiple different models (also called base learners) and makes a final prediction based on majority voting (for classification) or average prediction (for regression).
It helps improve accuracy by leveraging the strengths of several models instead of relying on just one.

In simple words:
A Voting Classifier takes predictions from many models and chooses the class that most models agree on (for classification) or the average value (for regression).

Types of Voting:

Hard Voting: Chooses the class that gets the majority of votes from the base models.

Soft Voting: Takes the average of predicted probabilities and selects the class with the highest probability.

In [79]:
from sklearn.ensemble import VotingClassifier

##### Hard Voting Classifier

In [80]:
base_learners = [
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('dt', DecisionTreeClassifier(random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
]

hard_voting_model = VotingClassifier(estimators=base_learners,voting='hard',n_jobs=-1)
hard_voting_model.fit(xtrain, ytrain)
hard_pred = hard_voting_model.predict(xtest)

In [81]:
print(confusion_matrix(ytest,hard_pred))
print(classification_report(ytest,hard_pred))

[[217  26]
 [ 52   5]]
              precision    recall  f1-score   support

          No       0.81      0.89      0.85       243
         Yes       0.16      0.09      0.11        57

    accuracy                           0.74       300
   macro avg       0.48      0.49      0.48       300
weighted avg       0.68      0.74      0.71       300



##### Soft Voting Classifier 

In [82]:
soft_voting_model = VotingClassifier(estimators=base_learners,voting='soft',n_jobs=-1)
soft_voting_model.fit(xtrain, ytrain)
y_pred_soft = soft_voting_model.predict(xtest)

In [83]:
print(confusion_matrix(ytest,y_pred_soft))
print(classification_report(ytest,y_pred_soft))

[[199  44]
 [ 44  13]]
              precision    recall  f1-score   support

          No       0.82      0.82      0.82       243
         Yes       0.23      0.23      0.23        57

    accuracy                           0.71       300
   macro avg       0.52      0.52      0.52       300
weighted avg       0.71      0.71      0.71       300

